# 🚁 Drone Semantic Segmentation with U-Net

**Training notebook** — Run on Google Colab (GPU) or Kaggle.

This notebook:
1. Downloads/loads the Semantic Drone Dataset
2. Defines a U-Net model from scratch
3. Trains for 30 epochs with GPU acceleration
4. Evaluates with Dice score & per-class IoU
5. Generates sample prediction visualizations
6. Saves model weights + training curves for download

---
**⚙️ Before running:** Go to `Runtime → Change runtime type → GPU (T4)`

## 1. Setup & Dependencies

In [ ]:
!pip install -q albumentations==1.3.1

import os
import numpy as np
import cv2
import csv
from glob import glob
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Load Dataset

**Kaggle:** Dataset is auto-mounted at `/kaggle/input/semantic-drone-dataset/`

**Colab:** Run the cell below to download from Kaggle (you'll need your Kaggle API key)

In [ ]:
# ========== PICK YOUR PLATFORM ==========

# --- OPTION A: KAGGLE (uncomment if on Kaggle) ---
# IMAGE_DIR = '/kaggle/input/semantic-drone-dataset/dataset/semantic_drone_dataset/original_images'
# MASK_DIR  = '/kaggle/input/semantic-drone-dataset/dataset/semantic_drone_dataset/label_images_semantic'

# --- OPTION B: COLAB (uncomment if on Colab) ---
# Run this once to download the dataset:
# !pip install -q kaggle
# from google.colab import files
# files.upload()  # Upload your kaggle.json API key
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d bulentsiyah/semantic-drone-dataset -p /content/data --unzip
# IMAGE_DIR = '/content/data/dataset/semantic_drone_dataset/original_images'
# MASK_DIR  = '/content/data/dataset/semantic_drone_dataset/label_images_semantic'

# --- AUTO-DETECT ---
if os.path.exists('/kaggle/input/semantic-drone-dataset'):
    IMAGE_DIR = '/kaggle/input/semantic-drone-dataset/dataset/semantic_drone_dataset/original_images'
    MASK_DIR  = '/kaggle/input/semantic-drone-dataset/dataset/semantic_drone_dataset/label_images_semantic'
    print('Kaggle detected!')
elif os.path.exists('/content/data/dataset'):
    IMAGE_DIR = '/content/data/dataset/semantic_drone_dataset/original_images'
    MASK_DIR  = '/content/data/dataset/semantic_drone_dataset/label_images_semantic'
    print('Colab detected!')
else:
    raise FileNotFoundError('Dataset not found. Please uncomment one of the options above.')

image_paths = sorted(glob(f'{IMAGE_DIR}/*.jpg') + glob(f'{IMAGE_DIR}/*.png'))
mask_paths  = sorted(glob(f'{MASK_DIR}/*.png'))

print(f'Images: {len(image_paths)}')
print(f'Masks:  {len(mask_paths)}')

## 3. Class Mapping (24 → 5 classes)

| ID | Group | What it includes |
|----|-------|------------------|
| 0 | **Obstacles** | buildings, walls, fences, rocks, etc. |
| 1 | **Water** | rivers, pools |
| 2 | **Nature** | grass, trees, vegetation |
| 3 | **Moving** | people, dogs, cars, bicycles |
| 4 | **Landable** | paved areas, gravel, flat roofs |

In [ ]:
NUM_CLASSES = 5
CLASS_NAMES = ['obstacles', 'water', 'nature', 'moving', 'landable']

GROUPED_CLASSES = {
    0: {0, 6, 10, 11, 12, 13, 14, 21, 22, 23},  # obstacles
    1: {5, 7},                                     # water
    2: {2, 3, 8, 19, 20},                          # nature
    3: {15, 16, 17, 18},                           # moving
    4: {1, 4, 9},                                  # landable
}

COLOR_MAP = {
    0: [155, 38, 182],    # obstacles → purple
    1: [14, 135, 204],    # water     → blue
    2: [124, 252, 0],     # nature    → green
    3: [255, 20, 147],    # moving    → pink
    4: [169, 169, 169],   # landable  → gray
}

def remap_mask(mask):
    """Remap original 24-class mask to 5 grouped classes."""
    new_mask = np.zeros_like(mask)
    for new_class, old_classes in GROUPED_CLASSES.items():
        for c in old_classes:
            new_mask[mask == c] = new_class
    return new_mask

print('Class mapping defined ✓')

## 4. Dataset & DataLoaders

In [ ]:
class SegmentationDataset(Dataset):
    """PyTorch Dataset for drone semantic segmentation."""

    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(self.mask_paths[idx], 0)
        mask = remap_mask(mask)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']

        return img, mask.long()


# Augmentations
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(),
    ToTensorV2(),
])

# Split 80/20
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.2, random_state=42
)

train_dataset = SegmentationDataset(train_imgs, train_masks, train_transform)
val_dataset   = SegmentationDataset(val_imgs, val_masks, val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} images ({len(train_loader)} batches)')
print(f'Val:   {len(val_dataset)} images ({len(val_loader)} batches)')

## 5. U-Net Model Architecture

```
Input (3×512×512)
  ├─ Encoder: Conv→BN→ReLU (×2) + MaxPool at each stage  [64→128→256→512]
  ├─ Bridge:  1024 channels
  └─ Decoder: ConvTranspose2d + Skip Connection + Conv→BN→ReLU (×2)  [512→256→128→64]
Output (5×512×512)
```

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive (Conv2d → BatchNorm → ReLU) blocks."""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    """Classic U-Net: 4 encoder stages, bridge, 4 decoder stages with skip connections."""
    def __init__(self, in_channels=3, num_classes=5):
        super().__init__()
        # Encoder
        self.d1 = DoubleConv(in_channels, 64)
        self.d2 = DoubleConv(64, 128)
        self.d3 = DoubleConv(128, 256)
        self.d4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)
        # Bridge
        self.bridge = DoubleConv(512, 1024)
        # Decoder
        self.u4 = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.c4 = DoubleConv(1024, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.c1 = DoubleConv(128, 64)
        # Output
        self.out = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(self.pool(d1))
        d3 = self.d3(self.pool(d2))
        d4 = self.d4(self.pool(d3))
        b  = self.bridge(self.pool(d4))
        u4 = self.c4(torch.cat([self.u4(b), d4], dim=1))
        u3 = self.c3(torch.cat([self.u3(u4), d3], dim=1))
        u2 = self.c2(torch.cat([self.u2(u3), d2], dim=1))
        u1 = self.c1(torch.cat([self.u1(u2), d1], dim=1))
        return self.out(u1)


model = UNet(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded on {device}')
print(f'Total parameters: {total_params:,} ({total_params/1e6:.1f}M)')

## 6. Metrics

In [ ]:
def dice_score(pred, target, num_classes=NUM_CLASSES):
    """Compute mean Dice coefficient across all classes."""
    pred = torch.argmax(pred, dim=1)
    dice = 0.0
    for cls in range(num_classes):
        pred_c = (pred == cls).float()
        target_c = (target == cls).float()
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        dice += (2 * intersection + 1e-6) / (union + 1e-6)
    return dice / num_classes


def per_class_iou(pred, target, num_classes=NUM_CLASSES):
    """Compute IoU for each class."""
    pred = torch.argmax(pred, dim=1)
    ious = {}
    for cls in range(num_classes):
        pred_c = (pred == cls).float()
        target_c = (target == cls).float()
        intersection = (pred_c * target_c).sum().item()
        union = (pred_c + target_c).clamp(0, 1).sum().item()
        ious[CLASS_NAMES[cls]] = round((intersection + 1e-6) / (union + 1e-6), 4)
    return ious

print('Metrics defined ✓')

## 7. Training Loop 🏋️

- **30 epochs** with Adam optimizer
- **ReduceLROnPlateau** scheduler (halves LR if val loss plateaus for 5 epochs)
- **Best model checkpointing** (saves only when val Dice improves)

In [ ]:
# Hyperparameters
EPOCHS = 30
LR = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# Tracking
best_dice = 0.0
history = []

print(f'Starting training for {EPOCHS} epochs...')
print('=' * 80)

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── Validate ──
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = criterion(preds, masks)
            val_loss += loss.item()
            val_dice += dice_score(preds, masks).item()

    # Compute averages
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
    current_lr = optimizer.param_groups[0]['lr']

    scheduler.step(val_loss)

    # Log
    history.append({
        'epoch': epoch,
        'train_loss': round(train_loss, 4),
        'val_loss': round(val_loss, 4),
        'val_dice': round(val_dice, 4),
        'lr': current_lr,
    })

    # Save best model
    marker = ''
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), 'best_model.pth')
        marker = ' ✓ BEST'

    print(
        f'Epoch {epoch:2d}/{EPOCHS} │ '
        f'Train Loss: {train_loss:.4f} │ '
        f'Val Loss: {val_loss:.4f} │ '
        f'Val Dice: {val_dice:.4f} │ '
        f'LR: {current_lr:.1e}{marker}'
    )

# Save final model
torch.save(model.state_dict(), 'final_model.pth')
print('=' * 80)
print(f'Training complete! Best Val Dice: {best_dice:.4f}')

## 8. Training Curves 📈

In [ ]:
epochs_list = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
val_dices = [h['val_dice'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_list, train_losses, label='Train Loss', marker='o', markersize=3)
ax1.plot(epochs_list, val_losses, label='Val Loss', marker='o', markersize=3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dice
ax2.plot(epochs_list, val_dices, label='Val Dice', marker='o', markersize=3, color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_title('Validation Dice Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 9. Save Training Log 📋

In [ ]:
with open('training_log.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=history[0].keys())
    writer.writeheader()
    writer.writerows(history)

print('Saved training_log.csv')

# Print summary table
print(f"\n{'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10} {'Val Dice':>10} {'LR':>10}")
print('-' * 52)
for h in history:
    print(f"{h['epoch']:>5} {h['train_loss']:>12.4f} {h['val_loss']:>10.4f} {h['val_dice']:>10.4f} {h['lr']:>10.1e}")

## 10. Per-Class IoU Evaluation 📊

In [ ]:
# Load best model for evaluation
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

# Accumulate IoU across entire val set
all_ious = {name: [] for name in CLASS_NAMES}

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        batch_ious = per_class_iou(preds, masks)
        for name, iou in batch_ious.items():
            all_ious[name].append(iou)

# Print per-class IoU
print('\n' + '=' * 40)
print('PER-CLASS IoU (on validation set)')
print('=' * 40)
mean_iou = 0
for name in CLASS_NAMES:
    avg_iou = np.mean(all_ious[name])
    mean_iou += avg_iou
    bar = '█' * int(avg_iou * 30)
    print(f'{name:>12}: {avg_iou:.4f}  {bar}')
mean_iou /= NUM_CLASSES
print(f'{"mIoU":>12}: {mean_iou:.4f}')
print('=' * 40)

## 11. Sample Predictions 🎨

In [ ]:
def predict_and_visualize(img_path, model, device):
    """Run inference on one image and return Original | Mask | Overlay."""
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess
    augmented = val_transform(image=img_rgb)
    input_tensor = augmented['image'].unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(input_tensor)
        pred = torch.argmax(output, dim=1).squeeze().cpu().numpy()

    # Resize back to original
    pred = cv2.resize(pred.astype(np.uint8), (img.shape[1], img.shape[0]),
                      interpolation=cv2.INTER_NEAREST)

    # Color mask
    color_mask = np.zeros_like(img_rgb)
    for cls, color in COLOR_MAP.items():
        color_mask[pred == cls] = color

    # Overlay
    overlay = cv2.addWeighted(img_rgb, 0.6, color_mask, 0.4, 0)

    return img_rgb, color_mask, overlay


# Pick 6 sample images from validation set
n = len(val_imgs)
sample_indices = [0, n//5, 2*n//5, 3*n//5, 4*n//5, n-1]
sample_paths = [val_imgs[i] for i in sample_indices]

os.makedirs('sample_predictions', exist_ok=True)

fig, axes = plt.subplots(len(sample_paths), 3, figsize=(20, 5 * len(sample_paths)))
titles = ['Original', 'Segmentation Mask', 'Overlay']

for row, img_path in enumerate(sample_paths):
    original, mask, overlay = predict_and_visualize(img_path, model, device)

    for col, (img, title) in enumerate(zip([original, mask, overlay], titles)):
        axes[row, col].imshow(img)
        axes[row, col].set_title(title, fontsize=14)
        axes[row, col].axis('off')

    # Save individual combined image
    display_h, display_w = 400, 600
    orig_resized = cv2.resize(original, (display_w, display_h))
    mask_resized = cv2.resize(mask, (display_w, display_h))
    overlay_resized = cv2.resize(overlay, (display_w, display_h))
    combined = np.hstack([orig_resized, mask_resized, overlay_resized])
    name = os.path.splitext(os.path.basename(img_path))[0]
    cv2.imwrite(f'sample_predictions/{name}_prediction.png',
                cv2.cvtColor(combined, cv2.COLOR_RGB2BGR))

plt.tight_layout()
plt.savefig('sample_predictions_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved sample predictions ✓')

## 12. Class Legend

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
for i, (name, color) in enumerate(zip(CLASS_NAMES, COLOR_MAP.values())):
    rgb = [c / 255 for c in color]
    ax.barh(i, 1, color=rgb, edgecolor='black', linewidth=1.5)
    ax.text(0.5, i, name.capitalize(), ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')

ax.set_yticks([])
ax.set_xticks([])
ax.set_title('Class Legend', fontsize=16, fontweight='bold')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig('class_legend.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Download Results ⬇️

Run this cell to download all results (works on Colab):

In [ ]:
import shutil

# Organize results
os.makedirs('results', exist_ok=True)
for f in ['training_curves.png', 'training_log.csv', 'class_legend.png', 'sample_predictions_grid.png']:
    if os.path.exists(f):
        shutil.copy(f, f'results/{f}')

if os.path.exists('sample_predictions'):
    shutil.copytree('sample_predictions', 'results/sample_predictions', dirs_exist_ok=True)

# Create zip for easy download
shutil.make_archive('drone_segmentation_results', 'zip', '.', 'results')
shutil.make_archive('drone_segmentation_weights', 'zip', '.', '.')

print('Created:')
print('  📦 drone_segmentation_results.zip  (curves, predictions, logs)')
print('  📦 drone_segmentation_weights.zip   (model .pth files)')

# Colab auto-download
try:
    from google.colab import files
    files.download('drone_segmentation_results.zip')
    files.download('best_model.pth')
    print('\n⬇️  Downloads started!')
except ImportError:
    print('\n📥 On Kaggle: go to Output tab to download files')

---

## ✅ Next Steps

After downloading:
1. Copy `best_model.pth` → your project's `weights/` folder
2. Copy `results/` → your project's `results/` folder
3. Push to GitHub!

```
drone-segmentation/
├── weights/best_model.pth    ← from this notebook
├── results/                  ← from this notebook
│   ├── training_curves.png
│   ├── training_log.csv
│   ├── class_legend.png
│   └── sample_predictions/
├── model.py
├── dataset.py
├── train.py
├── infer.py
├── config.py
├── utils.py
├── README.md
├── requirements.txt
└── .gitignore
```